

                          © 2026 NOOB DEV                                   
                       ALL RIGHTS RESERVED                                  
                                                                              
  This material is the **exclusive intellectual property of Noob Dev**.          
  **Strictly prohibited** to distribute, share, publish, reproduce, or           
  transmit this content — in whole or in part — outside of Noob Dev          
  sessions, platforms, or enrolled students.                                 
                                                                            
  Unauthorized sharing of this assignment outside the Noob Dev               
  community is a **violation of program terms** and may result in                
  immediate removal + legal actions from the Company.                                        

# CAIEP L1 : AI Personal Finance Tracker
## Noob Dev — Certified AI Engineer Professional L1

---

### ⏰ Deadline: **April 30, 2026 — 1 PM (SL TIME)**

---

This is your first real AI **agent** project. You are going to build a fully functional, chat-based personal finance tracker powered by a local AI agent with tool calling, a persistent SQLite database, and a Gradio user interface.

**This is not a tutorial.** You are expected to apply everything you have learned over the past two weeks and figure out implementation details on your own. That is how real engineering works.

> No OpenAI API key required. This project must run 100% free using **Groq**, **Ollama** or **OpenRouter** (If you want, you can go ahead with a paid OpenAI model, but it’s totally optional.).

---
## 🎯 Project Overview

You will build a **chat-first personal finance assistant** that users interact with entirely through natural language — no forms, no dropdowns, no spreadsheets. Just conversation.

The AI agent will:
- **Understand** what the user is saying (spending money, checking budget, etc.)
- **Decide** which tool to call based on the message
- **Execute** the tool (read or write to the database)
- **Respond** with a helpful, friendly message

### Core User Flows

| User says... | AI does... |
|---|---|
| `"My salary this month is $3,000"` | Calls `set_salary()` → saves to DB |
| `"Just paid $800 for rent"` | Calls `log_expense()` → saves to DB |
| `"Grabbed lunch for $12.50"` | Calls `log_expense()` → saves to DB |
| `"How much do I have left?"` | Calls `get_balance()` → reads DB → responds |
| `"Show me my spending breakdown"` | Calls `get_expense_summary()` → reads DB → responds |

### Example Conversation

```
User:  My monthly salary is $3,000
AI:    Got it! I've saved your April 2026 salary as $3,000.00.

User:  Paid rent today — $900
AI:    Logged $900.00 for Rent. Spent so far: $900.00 | Remaining: $2,100.00

User:  Bought groceries, spent $67.50 at the supermarket
AI:    Logged $67.50 for Groceries. Spent so far: $967.50 | Remaining: $2,032.50

User:  How much is left for the month?
AI:    Here's your April snapshot:
       Salary: $3,000.00 | Spent: $967.50 | Remaining: $2,032.50

User:  Break down my spending by category
AI:    April Spending Breakdown:
       - Rent:       $900.00  (30.0% of salary)
       - Groceries:  $67.50   (2.3% of salary)
       Total spent: $967.50 | Remaining: $2,032.50
```

---
## 🛠️ Tech Stack

| Component | Tool | Notes |
|-----------|------|-------|
| LLM | Groq (free tier), OpenRouter or Ollama (local) | Must support tool calling |
| UI | Gradio | Learned in Week 2  |
| Database | SQLite3 (built-in Python) | Learned in Week 2  |
| Tool Calling | OpenAI-compatible format | Learned in Week 2  |
| HTTP Client | OpenAI Python library | Same library, different `base_url` |

### Recommended Free Models

**Option A — Groq (recommended):**
- Sign up free at [console.groq.com](https://console.groq.com) — no credit card needed
- Model: `llama-3.3-70b-versatile`
- Fast, reliable, excellent tool calling support

**Option B — Ollama (fully local, no internet needed):**
- Install from [ollama.com](https://ollama.com)
- Run: `ollama pull llama3.1` in your terminal
- Model: `llama3.1` (use `llama3.1`, not `llama3.2` — tool calling support is better)

> ⚠️ **Important:** Not all models support tool calling. Use the recommended models above. If your tools are never being called, you are likely using the wrong model.

> ⚠️ **Important:** Add all the dependencies to the requirements.txt file.

---
## ⚙️ Step 1: Imports & Model Configuration

Set up your imports and configure which model/provider you are using.

---
## 🗄️ Step 1: Database Setup

Your SQLite database needs **two tables** to store salary and expense data.

### Table 1: `salary`
Stores the user's monthly income.

| Column | Type | Description |
|--------|------|-------------|
| `id` | INTEGER PRIMARY KEY | Auto-increments |
| `amount` | REAL | Monthly salary amount |
| `month` | TEXT | Month name — e.g. `"April"` |
| `year` | INTEGER | Year — e.g. `2026` |
| `created_at` | TEXT | Timestamp when recorded |

### Table 2: `expenses`
Stores every expense the user logs.

| Column | Type | Description |
|--------|------|-------------|
| `id` | INTEGER PRIMARY KEY | Auto-increments |
| `amount` | REAL | Expense amount |
| `category` | TEXT | e.g. `"Groceries"`, `"Transport"`, `"Rent"` |
| `description` | TEXT | Short description of the expense |
| `date` | TEXT | Date of the expense (`YYYY-MM-DD`) |
| `created_at` | TEXT | Timestamp when logged |


> 💡 **Hint:** You can use Gen AI tools for setup database and tables its totally fine only for this section.

---
## 🔧 Step 2: Implement the Tool Functions

You need **4 Python functions** that the AI agent can call. Each function must:
1. Connect to SQLite and do its job (read or write data)
2. Close the connection when done
3. Return a **string** result — the AI reads this string and uses it to compose its reply

---

### Tool 1 — `set_salary(amount, month, year)`
**When called:** User mentions their income, salary, or earnings for a month.
**What it does:** Inserts or replaces the salary record for that month in the database.
**Returns:** A confirmation string, e.g. `"Salary of $3,000.00 for April 2026 saved."`

---

### Tool 2 — `log_expense(amount, category, description, date)`
**When called:** User mentions spending money on anything.
**What it does:** Inserts a new expense row into the database. If `date` is not given, use today.
**Returns:** A confirmation string that also includes the updated balance — e.g. `"Logged $67.50 for Groceries. Spent: $967.50 | Remaining: $2,032.50"`

---

### Tool 3 — `get_balance()`
**When called:** User asks how much money is left, what their balance is, or how they are tracking.
**What it does:** Queries the salary and total expenses for the **current month**, calculates remaining.
**Returns:** A formatted string with salary, total spent, and remaining balance.

---

### Tool 4 — `get_expense_summary()`
**When called:** User asks for a breakdown, summary, or wants to know where their money went.
**What it does:** Groups all expenses for the current month by category, calculates totals and percentages.
**Returns:** A formatted string showing each category, amount, and percentage of salary.

> ⚠️ **Important rules for all functions:**
> - Always call `conn.close()` after you are done — never leave a connection open
> - Always call `conn.commit()` after any INSERT or UPDATE
> - Never put user values directly into SQL strings — always use `?` placeholders to prevent SQL injection

---
## 📋 Step 3: Describe Your Tools to the AI

The AI model does not know your Python functions exist. You have to describe them in a specific JSON format so the model knows:
- What tools are available
- When to use each one
- What parameters each tool expects

This is the **exact same format** used in **Week 2** for the airline support bot. Go back and look at that notebook — the `tools` list structure is identical.

Each tool description has three parts:
1. **`name`** — must exactly match your Python function name
2. **`description`** — tells the AI *when* to call this tool (write this carefully!)
3. **`parameters`** — describes each argument the function needs

> 💡 The `description` field is crucial. If you write a vague description, the AI will not know when to use the tool. Be specific about what triggers each tool call.

---
## 🔄 Step 4: The Tool Calling Loop

This is the **core logic** of your AI agent. When the user sends a message, your code must:

```
  [User message]
       │
       ▼
  Send to AI with tools list
       │
       ▼
  Did AI request a tool call?
  ├── YES → Extract tool name & args
  │          Call Python function
  │          Append result to messages
  │          Send back to AI  ──────────────┐
  │                                         │ (loop repeats)
  └── NO  → Return AI text response         │
             to user                        │
```

---
## 🖥️ Step 5: Build the Gradio UI

Your UI must use Gradio, and you can freely customize it as you like since it’s your project.

---
## 🧪 Test Conversations

Once your app is running, test it with all of the following inputs. Your app must handle every single one correctly for full marks.

### 1. Setting Up Income
```
My monthly salary is $3,500
I earn $4,000 a month
Set my income to 2800
```

### 2. Logging Expenses (natural language variety)
```
Spent $12 on coffee this morning
Paid rent — $900
Bought groceries for $67.50 at Walmart
Grabbed lunch with a friend, cost me $23
Netflix subscription renewed, $15.99
Filled up my gas tank, it was $55
Doctor appointment copay $30
```

### 3. Checking Balance
```
How much do I have left?
What's my remaining budget?
Am I on track this month?
How much have I spent so far?
```

### 4. Getting Summary
```
Show me my spending breakdown
Where am I spending the most?
Give me a summary of this month's expenses
What categories have I spent on?
```

> ✅ If all of these work, your agent is properly wired up.

---
## 📊 Evaluation Rubric

| Section | Marks | What We Are Looking For |
|---------|-------|-------------------------|
| Database setup | **10** | Tables created with correct columns and types |
| Tool implementations | **25** | All functions work correctly — read and write to DB accurately |
| Tool JSON definitions | **10** | All tools described with clear descriptions and correct parameter schemas |
| Tool calling loop | **20** | AI correctly decides which tool to call; loop handles multi-step calls and terminates properly |
| Gradio UI layout | **20** | Proper UI |
| Code quality | **10** | Clean, readable variable names; no dead code; no hardcoded secrets |
| README + submission | **5** | Clear setup instructions and working screenshot |
| **Total** | **100** | |

---
## 📤 Submission Instructions

**🗓 Deadline: April 30, 2026 — 1:00 PM (Sri Lanka Time)**  
Submissions after this time will **NOT be accepted under any circumstances**.

---

## 📦 Submission Components

You are required to submit **ALL of the following components**:

### 1️⃣ Full Project (ZIP File)

Submit a **single `.zip` file** with the exact structure below:

```
YourName_WhatsAppNumber_FinanceTracker/
├── your main application files (.ipynb)
├── your test database with some sample data in it
├── requirements.txt        ← all pip packages your project needs
└── README.md               ← setup and run instructions + screenshot

And all the necessary files, basically. 😉
```

---

### 2️⃣ 🎥 Video Demo (Mandatory)

You must create a **video demonstration** covering:

- Explanation of your **code structure**
- Key features and **logic of your implementation**
- **Live demo** of your application working end-to-end
- Keep it Under 10 min

> ⚠️ Make sure your video is clear, properly explained, and not rushed.

---

### 3️⃣ ☁️ Google Drive Submission

- Upload **both your ZIP file and Video Demo** to Google Drive  
- Set sharing permissions to: **“Anyone with the link can view”**  
- Double-check that your links are accessible without login issues  

---

### 4️⃣ 📝 Final Submission (Google Form)

- Submit your **Google Drive links** via the **Google Form provided**  
- Ensure:
  - Links are correct  
  - Files are accessible  
  - Submission is completed before the deadline  

---

## 🚫 Do NOT Include

- ❌ `.env` files or any **API keys / secrets**  
- ❌ `__pycache__` folders or unnecessary files  
- ❌ Large model files (e.g., Ollama models — keep them local)  
- ❌ Any irrelevant content  

---

## ⚠️ Submission Policy

> Late submissions will be **automatically rejected** after the deadline.  
> No extensions will be provided — plan and submit early.

```
╔══════════════════════════════════════════════════════════════════════════════╗
║  Good luck. Build something you would actually use.                         ║
║                                              — Noob Dev Team                ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

Import Libraries

In [1]:
import sqlite3
import os
import json
from datetime import datetime
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

#load env file
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER API key not found, please set it in the ,env file")

#model name
MODEL_NAME = "openai/gpt-oss-120b:free"

client = OpenAI(
    api_key= OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer" : "http://localhost",
        "X-title" : "Finance Tracker",
    }
)

#Database file path
DB_PATH = "finance.db"

print("Import done..")
print("API key leaded..")
print("CLient configured for", MODEL_NAME)
print("KEY:", OPENROUTER_API_KEY)

f:\AI Engineer Lab 1\chamuditha\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import done..
API key leaded..
CLient configured for openai/gpt-oss-120b:free
KEY: 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


Database Setup

In [2]:
def init_db():
    """
    Create the SQLite database and both tables if they don't exist.
    Safe to run multiple times witout losing data. 
    """
    conn = sqlite3.connect(DB_PATH)

    conn.execute('''
        CREATE TABLE IF NOT EXISTS salary (
                 id  INTEGER PRIMARY KEY AUTOINCREMENT,
                 amount      REAL      NOT NULL,
                 month       TEXT      NOT NULL,
                 year        INTEGER   NOT NULL,
                 created_at  TEXT      NOT NULL
                 )
    ''')

    conn.execute(''' 
        CREATE TABLE IF NOT EXISTS expenses (
                 id           INTEGER    PRIMARY KEY AUTOINCREMENT,
                 amount       REAL       NOT NULL,
                 category     TEXT       NOT NULL,
                 description  TEXT,
                 date         TEXT       NOT NULL,
                 created_at   TEXT       NOT NULL)
    ''')

    conn.commit()
    conn.close()
    print("Database ready : ",DB_PATH)

# run
init_db()

Database ready :  finance.db


Implement the Tool Functions

In [3]:
#Tool 1 -set_salary
# Called when :  user mention their income/ salary / earning
def set_salary(amount:float, month:str,year:int)->str:
    """ Save (or replace) the user's monthly salary in the database.

        We first delete any existing salary for that month/year so th user can correct a mistake without ending up with duplicate raws.

        Args:
            amount : The salary amount in dollers (e.g 3000.0),
            month  : Month name as a string (e.g "April"),
            year   : Four digit year (e.g 2026)
        
        Returns : 
            A Confirmation String with the AI will include in its reply.
    """
    conn = sqlite3.connect(DB_PATH)

    # Delete Any Previous salary for this month / year (allow correction)
    conn.execute(
            "DELETE FROM salary WHERE month = ? AND year = ?",
            (month,year)
     )
    
    # Insert the new salary row
    conn.execute(
            "INSERT INTO salary (amount,month,year,created_at) VALUES (?,?,?,?)",
            (amount, month,year, datetime.now().isoformat())
    )

    conn.commit()
    conn.close()

    return f"Salary of ${amount:,.2f} for {month} {year} has been saved."

#Tool 2 -  log_expense
def log_expense(amount:float,category:str,description:str,date:str="")->str:
    """
    Log a new expense and return an updated balance summery.

    Args:
        amount      : How much was spent in (e.g 67.50)
        category    : What category (e.g "Groceries", "Rent","Transport")
        description : Short note about the expense (e.g "Sathosa Supermarket")
        date        : Date as YYYY-MM-DD.  Defaults to today id empty.

    returns:
        A Confirmation String with current spent/remaining balance
    """
    if not date:
        date = datetime.now().strftime("%Y-%m-%d")
    conn = sqlite3.connect(DB_PATH)

    #Insert the expense
    conn.execute(
        "INSERT INTO expenses (amount,category,description,date,created_at) VALUES (?,?,?,?,?)",
        (amount, category, description, date, datetime.now().isoformat()),
    )
    conn.commit()

    # calculate running totlals for this to show the uyser their baalance
    current_month = datetime.now().strftime("%B")
    current_year = datetime.now().year

    salary_row = conn.execute(
        "SELECT amount FROM salary WHERE month = ? AND year = ?",
        (current_month, current_year)
    ).fetchone()

    total_spent_row = conn.execute(
        "SELECT COALESCE(SUM(amount),0) FROM expenses WHERE substr(date, 1, 7) = ?",
        (datetime.now().strftime('%Y-%m'),)

    ).fetchone()


    total_spent = total_spent_row[0]
    salary = salary_row[0] if salary_row else 0.0
    remaining = salary - total_spent

    result = f"Logged ${amount:,.2f} for {category}."

    if salary > 0:
        result += f"Spent so far : ${total_spent:,.2f} | Remaining : ${remaining:,.2f}"
    else:
        result += f"Total spent this month : ${total_spent:,.2f} (No salary set yet) "
    return result

# Tool 3 - get_summery
def get_balance() -> str:
    """ 
    Return a snapshot of this month's salary, total spent, and remaining balance.
    
    Returns:
          Formatted string with the monthly financial snapshot.
    """
    current_month = datetime.now().strftime("%B")
    current_year = datetime.now().year
    current_ym  = datetime.now().strftime("%Y-%m")

    conn = sqlite3.connect(DB_PATH)

    salary_row = conn.execute(
        "SELECT amount FROM salary WHERE month = ? AND year = ?",
        (current_month,current_year)
    ).fetchone()

    total_spent_row = conn.execute(
        "SELECT COALESCE(SUM(amount),0) FROM expenses WHERE substr(date, 1, 7) = ?",
        (datetime.now().strftime('%Y-%m'),)
    ).fetchone()

    conn.close()

    salary = salary_row[0] if salary_row else 0.0
    total_spent = total_spent_row[0]
    remaining = salary - total_spent

    if salary == 0:
        return (
            f"No salary recorded for {current_month} {current_year} yet."
            f"You have spent ${total_spent:,.2f} so far this month."
        )
    else:
        return (
            f"{current_month} {current_year} Snapshot:\n"
            f"Salary    : ${salary:,.2f}\n"
            f"Spent     : ${total_spent:,.2f}\n"
            f"Remaining : ${remaining:,.2f}" 
        )

# Tool 4 - get_expense_summary
# Called when :  user asks for a breakdown or summery of spending

# def get_expense_summery()->str:
#     """
#     Group this month's expenses by category and return a formatted summary string.

#     Returns:
#         A formatted string showing total spent per category breakdown.
#     """

#     current_month = datetime.now().strftime("%B")
#     current_year = datetime.now().year
#     current_ym  = datetime.now().strftime("%Y-%m")

#     conn = sqlite3.connect(DB_PATH)

#     salary_row = conn.execute(
#         "SELECT amount FROM salary WHERE month = ? AND year = ?",
#         (current_month,current_year)
#     ).fetchone()

#     #Group expenses by category for the current month
#     rows = conn.execute(
#         """ 
#         SELECT category, SUM(amount) AS total 
#         FROM expenses
#         WHERE substr(date, 1, 7) = ?
#         GROUP BY category
#         ORDER BY total DESC
#         """,current_ym
#     ).fetchall()

#     conn.close()

#     if not rows:
#         return f"No expenses recorded for {current_month} {current_year} yet."

#     salary = salary_row[0] if salary_row else 0.0
#     total_spent = sum(row[1] for row in rows)

#     lines = [f"{current_month} {current_year} Spending breakdown:"]

#     for category, total in rows:
#         pct = (total / salary * 100 if salary > 0 else 0.0)
#         lines.append(f" - {category:<15} ${total:,.2f} ({pct:,.1f}% of salary)")

#     lines.append(f"\n Total spent : ${total_spent:,.2f}")
#     if salary >0:
#         lines.append(f"Remaining balance : ${salary - total_spent:,.2f}")
#     lines.append(f"\n Total Spent : ${total_spent:,.2f}")
#     if salary > 0:
#         lines.append(f"Remaining balance : ${salary - total_spent:,.2f}")
    
#     return "\n".join(lines)



def get_expense_summary() -> str:
    current_month = datetime.now().strftime("%B")
    current_year  = datetime.now().year

    conn = sqlite3.connect(DB_PATH)

    salary_row = conn.execute(
        "SELECT amount FROM salary WHERE month = ? AND year = ?",
        (current_month, current_year)
    ).fetchone()

    ym = datetime.now().strftime("%Y-%m")

    rows = conn.execute(
        """
        SELECT category, SUM(amount) AS total
        FROM expenses
        WHERE substr(date, 1, 7) = ?
        GROUP BY category
        ORDER BY total DESC
        """,
        [ym]
    ).fetchall()

    conn.close()

    if not rows:
        return f"No expenses recorded for {current_month} {current_year} yet."

    salary      = salary_row[0] if salary_row else 0.0
    total_spent = sum(row[1] for row in rows)

    lines = [f"{current_month} {current_year} Spending Breakdown:"]
    for category, total in rows:
        pct = (total / salary * 100) if salary > 0 else 0.0
        lines.append(f"  - {category:<15} ${total:,.2f}  ({pct:.1f}% of salary)")

    lines.append(f"\n  Total Spent : ${total_spent:,.2f}")
    if salary > 0:
        lines.append(f"  Remaining   : ${salary - total_spent:,.2f}")

    return "\n".join(lines)

print("✅ get_expense_summery fixed!")

print("Tools defined..")

✅ get_expense_summery fixed!
Tools defined..


Describe Tool For AI

In [4]:
# This list is passed directly to the OpenAI/Groq API as the `tools` parameter.
# The structure is: type → function → name / description / parameters

tools = [
    {
        "type": "function",
        "function": {
            "name": "set_salary",
            "description": (
                "Save the user's monthly salary or income to the database. "
                "Call this tool whenever the user mentions their salary, income, earnings, "
                "paycheck, or monthly budget (e.g. 'my salary is $3000', "
                "'I earn $4000 a month', 'set my income to 2800')."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {
                        "type": "number",
                        "description": "The salary amount as a number (e.g. 3000.0)"
                    },
                    "month": {
                        "type": "string",
                        "description": "Month name in full, e.g. 'April'. Default to the current month if not specified."
                    },
                    "year": {
                        "type": "integer",
                        "description": "Four-digit year, e.g. 2026. Default to the current year if not specified."
                    }
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_expense",
            "description": (
                "Log a new expense to the database. "
                "Call this tool whenever the user mentions spending, paying, buying, "
                "or purchasing something (e.g. 'paid rent $900', 'bought groceries for $67', "
                "'spent $12 on coffee', 'Netflix renewed $15.99'). "
                "Infer the category from context if not explicitly stated."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {
                        "type": "number",
                        "description": "The expense amount as a number (e.g. 67.50)"
                    },
                    "category": {
                        "type": "string",
                        "description": (
                            "Expense category. Infer from context. "
                            "Common categories: Rent, Groceries, Transport, Food, "
                            "Entertainment, Health, Utilities, Subscriptions, Other."
                        )
                    },
                    "description": {
                        "type": "string",
                        "description": "Short description of what the expense was for (e.g. 'Walmart grocery run')"
                    },
                    "date": {
                        "type": "string",
                        "description": "Date of the expense in YYYY-MM-DD format. Leave empty to use today's date."
                    }
                },
                "required": ["amount", "category", "description"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": (
                "Get the current month's financial balance: salary, total spent, and remaining amount. "
                "Call this tool when the user asks how much money is left, what their balance is, "
                "how much they have remaining, or how they are tracking this month "
                "(e.g. 'how much do I have left?', 'what's my remaining budget?', "
                "'am I on track?', 'how much have I spent?')."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": (
                "Get a breakdown of this month's spending grouped by category. "
                "Call this tool when the user asks for a summary, breakdown, or analysis of their expenses "
                "(e.g. 'show me my spending breakdown', 'where am I spending the most?', "
                "'give me a summary', 'what categories have I spent on?')."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

print(f"Defined {len(tools)} tools for the AI agent.")

Defined 4 tools for the AI agent.


The Tool Calling loop

In [5]:
# Map tool names (strings) → actual Python functions
# This lets us call the right function dynamically based on what the AI requests
TOOL_MAP = {
    "set_salary"           :   set_salary,
    "log_expense"          :   log_expense,
    "get_balance"          :   get_balance,
    "get_expense_summary"  : get_expense_summary,
}


# The system prompt tells the AI what its role is and how to behave
SYSTEM_PROMPT = """ 
                You are a friendly and helpful personal finance assistant.
                You help users track their monthly income and expenses through natural conversaton.

                When a user mention their salary or income, call set_salary.
                When a user mentions spending money on anything, call log_expense.
                When a user asks how much money is left or their balance, call get_balance.
                When a user asks for a spending breakdown or summery, call get_expense_summary.

                Always be warm, encouraging, and concise in your replies.
                After calling a Fool, Use the resulf to give the user a helpful, natural language response.
                Use today's date for the current month/year when not specified by the user.
                Today's date : """ + datetime.now().strftime("%B %d, %Y") + """
                 """

def chat(user_message: str, history: list) :

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for turn in history:
        if isinstance(turn, dict):
            messages.append(turn)
        else:
            messages.append({"role": "user", "content": turn[0]})
            messages.append({"role": "assistant", "content": turn[1]})

    messages.append({"role": "user", "content": user_message})

    
    # STEP 1: TOOL CALL LOOP (NON-STREAM)

    while True:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=tools,
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)

            for tool_call in msg.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                tool_fn   = TOOL_MAP.get(tool_name)

                if tool_fn:
                    tool_result = tool_fn(**tool_args)
                else:
                    tool_result = f"Unknown tool: {tool_name}"
                    tool_result = f"Unknown tool: {tool_name}"

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                })
        else:
            messages.append(msg)
            break


    # STEP 2: STREAM FINAL RESPONSE  
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        stream=True,
    )

    partial = ""

    for chunk in stream:
        if chunk.choices[0].delta and chunk.choices[0].delta.content:
            partial += chunk.choices[0].delta.content
            yield partial



print("Agent loop ready.")

Agent loop ready.


Gradio UI

In [6]:

# Build the Gradio chat interface

demo = gr.ChatInterface(
    fn=chat,  # our agent function
    title="AI Personal Finance Tracker",
    description=(
        "Chat naturally about your money. Tell me your salary, log expenses, "
        "and ask for your balance or spending breakdown — all in plain English."
    ),
    examples=[
        "My monthly salary is $3,500",
        "Paid rent — $900",
        "Bought groceries for $67.50 at Walmart",
        "How much do I have left this month?",
        "Show me my spending breakdown",
        "Spent $12 on coffee this morning",
        "Netflix subscription renewed, $15.99",
    ],
)

# Launch the app
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://64fe0d93a97c4ee58a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
